✅ 4. ParentDocumentRetriever

Use Case: Retrieve parent chunks based on a retrieval of smaller child chunks, great for keeping context intact in long documents.

In [4]:
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.vectorstores import FAISS
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from langchain_core.stores import InMemoryStore


# 1. Prepare documents
docs = [Document(page_content="LangChain helps build LLM-powered apps with memory and agents.", metadata={"id": "1"}),
        Document(page_content="Agents in LangChain use tools to answer questions.", metadata={"id": "2"})]

# 2. Setup child splitter
child_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

# 3. Setup vectorstore for children
embedding = OllamaEmbeddings(
    model = 'nomic-embed-text'
)
vectorstore = FAISS.from_documents(docs, embedding)

# 4. Parent retriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=InMemoryStore(),  # Stores parent docs
    child_splitter=child_splitter
)

# 5. Add documents
retriever.add_documents(docs)

# 6. Retrieve
results = retriever.invoke("What are agents?")
for doc in results:
    print("📄 Retrieved Doc:", doc.page_content)


📄 Retrieved Doc: LangChain helps build LLM-powered apps with memory and agents.
📄 Retrieved Doc: Agents in LangChain use tools to answer questions.


In [12]:
# 1. Prepare documents
docs = [Document(page_content="LangChain helps build LLM-powered apps with memory and agents.", metadata={"id": "1"}),
        Document(page_content="Agents in LangChain use tools to answer questions.", metadata={"id": "2"})]

print(docs)

[Document(metadata={'id': '1'}, page_content='LangChain helps build LLM-powered apps with memory and agents.'), Document(metadata={'id': '2'}, page_content='Agents in LangChain use tools to answer questions.')]


In [16]:
from langchain_community.document_loaders import WebBaseLoader
# pip install beautifulsoup4

# 1. Load website
loader = WebBaseLoader("https://python.langchain.com/docs/")
docs = loader.load()


# 2. Setup child splitter
child_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

# 3. Setup vectorstore for children
embedding = OllamaEmbeddings(
    model = 'nomic-embed-text'
)
vectorstore = FAISS.from_documents(docs, embedding)

# 4. Parent retriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=InMemoryStore(),  # Stores parent docs
    child_splitter=child_splitter
)

# 5. Add documents
retriever.add_documents(docs)

# 6. Retrieve
results = retriever.invoke("What are agents?")
for doc in results:
    print("📄 Retrieved Doc:", doc.page_content)

📄 Retrieved Doc: LangChain overview - Docs by LangChainSkip to main contentDocs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewMarkdown MessagesTool CallingHuman-in-the-LoopBranching ChatReasoning TokensStructured OutputMessage QueuesJoin & Rejoin StreamsTime TravelGenerative UIUI Library IntegrationsAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIDeploy with LangSmithDeploymentObservabilityOn this page Create an agent Core benefitsLangChain overviewCopy pageLangChain is an open source framework with a prebuilt

✅ 5. BM25Retriever

Use Case: Pure keyword-based search (like traditional search engines). No embeddings needed.

In [ ]:
from langchain_classic.retrievers import BM25Retriever
from langchain_core.documents import Document

# Create simple text docs
docs = [
    Document(page_content="LangChain enables LLM applications."),
    Document(page_content="Vector search is powerful."),
    Document(page_content="BM25 is a classical retrieval method.")
]

# Create BM25 retriever
bm25_retriever = BM25Retriever.from_documents(docs)

# Retrieve
results = bm25_retriever.invoke("How does BM25 work?")
for doc in results:
    print("📝 BM25 Result:", doc.page_content)


📝 BM25 Result: BM25 is a classical retrieval method.
📝 BM25 Result: Vector search is powerful.
📝 BM25 Result: LangChain enables LLM applications.


✅ 6. EnsembleRetriever

Use Case: Combine multiple retrievers (e.g., keyword + vector-based) with weighted scores.

In [8]:
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever
from langchain_core.documents import Document
from langchain_classic.vectorstores import FAISS

# Sample docs
docs = [Document(page_content="LangChain supports LLMs."), Document(page_content="You can build AI apps using LangChain.")]

# BM25 Retriever
bm25 = BM25Retriever.from_documents(docs)

# Vector Retriever
embedding = OllamaEmbeddings(model='nomic-embed-text')
vectorstore = FAISS.from_documents(docs, embedding)
vector_retriever = vectorstore.as_retriever()

# Ensemble Retriever (equal weight)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25, vector_retriever],
    weights=[0.5, 0.5]
)

# Query
results = ensemble_retriever.invoke("AI apps using LangChain")
for doc in results:
    print("🔍 Ensemble Doc:", doc.page_content)


🔍 Ensemble Doc: You can build AI apps using LangChain.
🔍 Ensemble Doc: LangChain supports LLMs.


7.TimeWeightedVectorStoreRetriever

#This retriever boosts document relevance by factoring recency and importance of interactions (used in agents with memory or relevance ranking).

In [11]:
#7.TimeWeightedVectorStoreRetriever
#This retriever boosts document relevance by factoring recency and importance of interactions (used in agents with memory or relevance ranking).


from langchain_classic.retrievers import TimeWeightedVectorStoreRetriever
from langchain_core.documents import Document
from langchain_classic.vectorstores import FAISS


from datetime import datetime

# Initialize embedding & vectorstore
embedding = OllamaEmbeddings(model='nomic-embed-text')
docs = [
    Document(page_content="LangChain is for LLM-based apps", metadata={"last_accessed_at": datetime.now()}),
    Document(page_content="Vector search improves relevance", metadata={"last_accessed_at": datetime.now()})
]
vectorstore = FAISS.from_documents(docs, embedding)

# TimeWeighted Retriever
retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore,
    decay_rate=0.01,
    k=2,
    score_threshold=None
)

# Retrieve
results = retriever.invoke("What is LangChain?")
for r in results:
    print(r)
    print(r.page_content)
